In [1]:
import yaml
import torch
import seaborn as sns
from datetime import datetime
from DRF.BayesianOptimiser import BayesianOptimiser
from DRF.data_utils import prepare_tensor_datasets_ABC
from DRF.models import initialize_model
import pandas as pd

import torch
import os
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeat
from pyproj import Transformer
from global_land_mask import globe
import ast

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt



def EASE2toWGS84(x, y, return_vals="both", lon_0=0, lat_0=90):

    valid_return_vals = ['both', 'lon', 'lat']
    assert return_vals in ['both', 'lon', 'lat'], f"return_val: {return_vals} is not in valid set: {valid_return_vals}"
    EASE2 = f"+proj=laea +lon_0={lon_0} +lat_0={lat_0} +x_0=0 +y_0=0 +ellps=WGS84 +towgs84=0,0,0,0,0,0,0 +units=m +no_defs"
    WGS84 = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs"
    transformer = Transformer.from_crs(EASE2, WGS84)
    lon, lat = transformer.transform(x, y)
    if return_vals == "both":
        return lon, lat
    elif return_vals == "lon":
        return lon
    elif return_vals == "lat":
        return lat

/home/mhen/miniconda3/envs/max_project1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO (geometric_kernels): Numpy backend is enabled. To enable other backends, don't forget to `import geometric_kernels.*backend name*`.
INFO (geometric_kernels): We may be suppressing some logging of external libraries. To override the logging policy, call `logging.basicConfig`.
/home/mhen/miniconda3/envs/max_project1/lib/python3.10/site-packages/spherical_harmonics/fundamental_set.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
INFO (geometric_kernels): Torch backend enabl

In [2]:
#first get all track nos
tracks_dir = '/home/mhen/data/cs2s3/per_track/jan20/drf/week_interp/cs2s3'
filenames = os.listdir(tracks_dir)

tracknos = []
for file in filenames:
    trackno = file.split('track')[-1].split('.')[0] #isolate number
    tracknos.append(trackno)

#there should be two of each number - keep one of each duplicate value
dup_tracks = {track for track in tracknos if tracknos.count(track) == 2}
dup_tracks = sorted(dup_tracks, key=int)

In [3]:
def read_config(path):
    with open(path,'r') as f:
        config = yaml.safe_load(f)
    return config
def add_track_to_name(dir, filename,track_no,expand=True):
    if expand == True:
        results_dir = dir.replace('~','/home/mhen')
    elif expand == False:
        results_dir = dir
    name = filename.split('.')[0]
    ext = filename.split('.')[1]

    path = f'{results_dir}{name}_track{track_no}.{ext}'
    print(path)
    return path
#define target month as string
target_month = "2020-01"

def interpolate(config, tracks, results_dir):

    obs_data_path = config["data"]["obs_data_dir"] + config['data']['obs_data_name']
    test_data_path = config["data"]["obs_data_dir"] + config['data']['test_data_name']
    obs_extension = config['data']['obs_data_ext']
    test_extension = config['data']['test_data_ext']

    for i, track in enumerate(dup_tracks):
        print(f'Checking track {track}, {i+1} of {len(dup_tracks)}')

        #create all paths here, inserting tracknumber
        obs_path = obs_data_path + track + obs_extension
        test_path = test_data_path + track + test_extension

        #read the date string in the test dataframe
        #pass this date as the prediction date
        try:
            test_df = pd.read_csv(test_path)
        except FileNotFoundError:
            print('File not found, skipping...')
            continue
        date_str = test_df['date_string'].iloc[0]
        print(date_str)
        #interpolate only those in the month of the target result
        #to avoid edge cases
        if target_month in date_str:
            print(f'Track within target month {target_month}, proceeding...')

            results_dir = config['results']['results_dir']

            csv_path=add_track_to_name(
                results_dir, config['results']['csv_filename'],track)
            predictions_path=add_track_to_name(
                results_dir, config['results']['predictions_filename'],track)
            variance_path=add_track_to_name(
                results_dir, config['results']['variance_filename'],track)
            individual_predictions_path=add_track_to_name(
                results_dir, config['results']['individual_predictions_filename'],track)

            print(csv_path)

            #check files don't already exist
            path_list = [csv_path, predictions_path, variance_path, individual_predictions_path]
            if all(os.path.exists(p) for p in path_list):
                print(f'Track {track} already interpolated, skipping...')
                continue
            print(f'Track {track} files not found, proceeding with interpolation...')

        
            #split the data for this track ready to train
            spatial_X_train, temporal_X_train, y_train, spatial_X_test, temporal_X_test = (
                prepare_tensor_datasets_ABC(
                    obs_path, test_path, date_str))

            #create model
            device = torch.device(config["device"])


            def model_init_func(
                spatial_lengthscale, temporal_lengthscale, amplitude, device=device
                ):
                return initialize_model(
                    model_name=config["model"]["name"],
                    num_layers=config["model"]["num_layers"],
                    spatial_input_dim=spatial_X_train.shape[-1],
                    temporal_input_dim=temporal_X_train.shape[-1],
                    hidden_dim=config["model"]["hidden_dim"],
                    bottleneck_dim=config["model"]["bottleneck_dim"],
                    output_dim=config["model"]["output_dim"],
                    spatial_lengthscale=spatial_lengthscale,
                    temporal_lengthscale=temporal_lengthscale,
                    amplitude=amplitude,
                    device=device,
                    spatial_layer_type=config["model"]["spatial_layer_type"],
                    temporal_layer_type=config["model"]["temporal_layer_type"],
                    model_kwargs=config["model"].get("kwargs", {}),
                )

            #create optimizer
            optimizer = BayesianOptimiser(
                config,
                model_init_func,
                optimize_observation_noise=config["bayesian_optimization"][
                    "optimize_observation_noise"
                ],
                penalty_weight=config["bayesian_optimization"]["penalty_weight"],
            )

            #now train
            best_hyperparams, best_loss = optimizer.optimize(
                spatial_X_train, temporal_X_train, y_train, spatial_X_test, temporal_X_test
            )

            print(f"Best Hyperparameters: {best_hyperparams.cpu().numpy()}, Loss: {best_loss}")

            top_prediction = sorted(
                optimizer.test_predictions_per_iteration, key=lambda x: x[1]
            )[:1]

            def extract_tensors_and_params(pred_tuple):
                return pred_tuple[0], pred_tuple[2]

            def process_prediction_set(tensor_tuple):
                return torch.stack(tensor_tuple)

            extracted_predictions_and_params = [
                extract_tensors_and_params(pred) for pred in top_prediction
            ]
            processed_predictions = [
                process_prediction_set(pred[0]) for pred in extracted_predictions_and_params
            ]
            print("Shape of processed predictions:", processed_predictions[0].shape)
            final_test_predictions = processed_predictions[0].mean(dim=0)
            var_final_pred = processed_predictions[0].var(dim=0)

            nll_hyperparams_list = []
            for pred, nll, hyperparams in top_prediction:
                entry = {"NLL": nll}
                if isinstance(hyperparams, dict):
                    entry.update(hyperparams)
                elif isinstance(hyperparams, (list, tuple)):
                    for i, param in enumerate(hyperparams):
                        entry[f"param_{i}"] = param.item() if torch.is_tensor(param) else param
                else:
                    entry["param"] = hyperparams
                nll_hyperparams_list.append(entry)

            df = pd.DataFrame(nll_hyperparams_list)
            df.to_csv(csv_path, index=False)
            print(f"NLL and hyperparameters saved to {csv_path}")

            torch.save(final_test_predictions, predictions_path)
            torch.save(var_final_pred, variance_path)
            torch.save(
                processed_predictions[0], individual_predictions_path
            )
            print(f"Final test predictions saved for track {track}, {i+1} of {len(dup_tracks)} .")
        else:
            print(f'Track not within target month {target_month}, skipping...')
    print('All tracks interpolated!')

def get_results(config, tracks,results_dir):
    full_df = pd.DataFrame()
    csv_df = pd.DataFrame()
    print(f'Extracting results for  {len(tracks)} tracks...')
    for i, track in enumerate(tracks):
        print(f'Track {track}, {i+1} of {len(tracks)}...')
        test_data_path = config["data"]["obs_data_dir"] + config['data']['test_data_name']
        test_extension = config['data']['test_data_ext']
        test_path = test_data_path + str(track) + test_extension
        test_df = pd.read_csv(test_path)
        print(test_path)
        print(test_df.head())


        results_files = os.listdir(results_dir)
        results = [file for file in results_files if str(track) in file]
        print('results files:', results)
        predicted_mean_filename = f'final_predictions_track{track}.pt'
        predicted_var_filename = f'final_variance_track{track}.pt'
        indiv_filename = f'individual_final_predictions_track{track}.pt'
        results_csv_filename = f'results_track{track}.csv'
        pred_df_filename = f'interp_pt_test_track{track}.csv'
        predicted_mean = torch.load(results_dir + predicted_mean_filename)
        predicted_var = torch.load(results_dir + predicted_var_filename)
        #individual_final_predictions = torch.load(results_dir + results[3])
        
        data_dir = config['data']['obs_data_dir']
        pred_df = pd.read_csv(data_dir + pred_df_filename)
        pred_df['dist_along_track'] = test_df['dist_along_track']

        print(pred_df.shape) 
        print(pred_df.head())
        print(pred_df.isnull().sum())

        average_predictions = predicted_mean
        average_predictions = torch.Tensor(average_predictions)
        predicted_var = torch.Tensor(predicted_var)
        predicted_var_np = predicted_var.cpu().numpy().flatten()
        predicted_var_np = predicted_var_np.flatten()
        print(average_predictions.shape)
        average_predictions_np = average_predictions.cpu().numpy().flatten()
        average_predictions_np = average_predictions_np.flatten()

        #investigate other outputs
        indiv_final_preds = torch.load(results_dir + indiv_filename)
        #get hyperparameters
        results_df = pd.read_csv(results_dir + results_csv_filename)
        nll = results_df['NLL']
        param = results_df['param'].iloc[0]
        spatial = param.split(',')[0].split('[')[1]
        temporal = param.split(',')[1].split(' ')[1]
        amplitude = param.split(',')[2].split(' ')[1]
        noise = param.split(',')[3].split(']')[0].split(' ')[1]

        hyperparams_df = pd.DataFrame(
            { 'track' : track,
            'nll' : nll,
            'spatial' : spatial,
            'temporal' : temporal,
            'amplitude': amplitude,
            'noise' : noise}
        )

        indiv_pred = torch.Tensor(indiv_final_preds)
        

        csv_df = pd.concat([csv_df,hyperparams_df])

        pred_df['f*'] = average_predictions_np
        pred_df['f*_var'] = predicted_var_np
        pred_df['f*_std'] = np.sqrt(pred_df['f*_var'])
        pred_df["is_in_ocean"] = globe.is_ocean(pred_df['lat'], pred_df['lon'])
        pred_df = pred_df.loc[pred_df['is_in_ocean']]

        #put in uncertainties
        pred_df['upper'] = pred_df['f*'] + 1.96*pred_df['f*_std']
        pred_df['lower'] = pred_df['f*'] - 1.96*pred_df['f*_std']

        full_df = pd.concat([full_df,pred_df])

    return full_df, csv_df


In [4]:
#import config args
config_path = 'path/to/example_config.conf'
config_cs2s3 = read_config(config_path)
interpolate(config_cs2s3, dup_tracks, '/results/save/dir/)
cs2s3_full_df, cs2s3_csv_df = get_results(config_cs2s3, dup_tracks,'/results/save/dir/')

Checking track 1250, 1 of 278
2020-01-01
Track within target month 2020-01, proceeding...
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1250.csv
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_predictions_track1250.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_variance_track1250.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/individual_final_predictions_track1250.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1250.csv
Track 1250 already interpolated, skipping...
Checking track 1252, 2 of 278
2020-01-01
Track within target month 2020-01, proceeding...
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1252.csv
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_predictions_track1252.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_de

2020-01-07
Track within target month 2020-01, proceeding...
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1551.csv
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_predictions_track1551.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_variance_track1551.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/individual_final_predictions_track1551.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1551.csv
Track 1551 already interpolated, skipping...
Checking track 1553, 71 of 278
2020-01-07
Track within target month 2020-01, proceeding...
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/results_track1553.csv
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_predictions_track1553.pt
/home/mhen/DeepRandomFeatures/notebooks/per_track_demo/results/cs2s3_1week/final_

In [5]:
#save
cs2s3_full_df.to_csv('/path/to/results')